# Data Preprocessing and Feature Extraction for Classical ML
## Distributed Deep Learning for Smart Healthcare Diagnosis

This notebook demonstrates:
1. Data loading and exploration
2. Data cleaning and preprocessing (Pandas, NumPy)
3. Feature extraction from X-ray images for classical ML models
4. Handling missing values and data encoding
5. Feature scaling

**Dataset**: Chest X-Ray Pneumonia Dataset

In [ ]:
# Import Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import os
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Set visualization style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print("✓ All libraries imported successfully")
print(f"NumPy version: {np.__version__}")
print(f"Pandas version: {pd.__version__}")

## 1. Dataset Overview and Problem Understanding

**Real-World Problem**: Manual pneumonia diagnosis from chest X-rays is time-consuming and requires expert radiologists.

**ML Problem Definition**:
- **Input Data**: Chest X-ray images (224x224 pixels, grayscale)
- **Output Prediction**: Binary classification (NORMAL or PNEUMONIA)
- **Target Users**: Medical professionals, radiologists, healthcare facilities

**Design Thinking**:
- Automated screening system for faster diagnosis
- Assist doctors in remote areas with limited expertise
- Scalable solution for multiple healthcare institutions

In [ ]:
# Dataset Configuration
# Note: Update these paths according to your dataset location
DATA_DIR = Path('../data/chest_xray')  # Update this path
TRAIN_DIR = DATA_DIR / 'train'
TEST_DIR = DATA_DIR / 'test'
VAL_DIR = DATA_DIR / 'val'

# For demonstration, we'll create a sample dataset structure
print("=" * 60)
print("DATASET STRUCTURE")
print("=" * 60)
print(f"Expected Dataset Location: {DATA_DIR}")
print("\nExpected Structure:")
print("chest_xray/")
print("├── train/")
print("│   ├── NORMAL/")
print("│   └── PNEUMONIA/")
print("├── test/")
print("│   ├── NORMAL/")
print("│   └── PNEUMONIA/")
print("└── val/")
print("    ├── NORMAL/")
print("    └── PNEUMONIA/")
print("\n" + "=" * 60)

## 2. Data Loading and Exploration

Since the actual dataset may not be present, we'll create a demonstration with sample data loading functions.

In [ ]:
def load_image_paths(data_dir, max_samples=None):
    """Load image paths and labels from directory structure"""
    image_paths = []
    labels = []
    
    if not data_dir.exists():
        print(f"⚠️ Warning: Directory {data_dir} not found")
        print("Creating sample data structure for demonstration...")
        return [], []
    
    for class_name in ['NORMAL', 'PNEUMONIA']:
        class_dir = data_dir / class_name
        if class_dir.exists():
            files = list(class_dir.glob('*.jpeg')) + list(class_dir.glob('*.jpg')) + list(class_dir.glob('*.png'))
            if max_samples:
                files = files[:max_samples]
            image_paths.extend(files)
            labels.extend([class_name] * len(files))
    
    return image_paths, labels

# Try to load actual dataset
train_paths, train_labels = load_image_paths(TRAIN_DIR, max_samples=1000)
test_paths, test_labels = load_image_paths(TEST_DIR, max_samples=200)

if len(train_paths) > 0:
    print(f"✓ Loaded {len(train_paths)} training images")
    print(f"✓ Loaded {len(test_paths)} test images")
else:
    print("⚠️ Dataset not found. Creating synthetic sample data for demonstration...")
    # Create synthetic data for demonstration
    np.random.seed(42)
    n_samples = 500
    train_paths = [f"sample_image_{i}.jpg" for i in range(n_samples)]
    train_labels = ['NORMAL'] * (n_samples//2) + ['PNEUMONIA'] * (n_samples//2)
    test_paths = [f"test_image_{i}.jpg" for i in range(100)]
    test_labels = ['NORMAL'] * 50 + ['PNEUMONIA'] * 50
    print(f"✓ Created {len(train_paths)} synthetic training samples")
    print(f"✓ Created {len(test_paths)} synthetic test samples")

## 3. Data Cleaning and Pandas DataFrame Creation

Convert image data into structured Pandas DataFrame for analysis and classical ML processing.

In [ ]:
# Create DataFrame with image information
train_df = pd.DataFrame({
    'image_path': train_paths,
    'label': train_labels
})

test_df = pd.DataFrame({
    'image_path': test_paths,
    'label': test_labels
})

print("Training Dataset Information:")
print("=" * 60)
print(train_df.info())
print("\nClass Distribution:")
print(train_df['label'].value_counts())
print("\nClass Distribution (%):")
print(train_df['label'].value_counts(normalize=True) * 100)

# Check for missing values
print("\n" + "=" * 60)
print("DATA QUALITY CHECK")
print("=" * 60)
print(f"Missing values in training data: {train_df.isnull().sum().sum()}")
print(f"Missing values in test data: {test_df.isnull().sum().sum()}")
print(f"Duplicate entries in training data: {train_df.duplicated().sum()}")

# Handle missing values (if any)
train_df = train_df.dropna()
test_df = test_df.dropna()
print("✓ Data cleaning completed")

## 4. Exploratory Data Analysis (EDA)

In [ ]:
# Visualize class distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Training data distribution
train_df['label'].value_counts().plot(kind='bar', ax=axes[0], color=['skyblue', 'coral'])
axes[0].set_title('Training Data - Class Distribution', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Class', fontsize=12)
axes[0].set_ylabel('Number of Images', fontsize=12)
axes[0].tick_params(rotation=0)

# Test data distribution
test_df['label'].value_counts().plot(kind='bar', ax=axes[1], color=['skyblue', 'coral'])
axes[1].set_title('Test Data - Class Distribution', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Class', fontsize=12)
axes[1].set_ylabel('Number of Images', fontsize=12)
axes[1].tick_params(rotation=0)

plt.tight_layout()
plt.savefig('../artifacts/class_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("✓ Visualization saved to artifacts/class_distribution.png")

## 5. Feature Extraction from Images

For classical ML algorithms to work with images, we need to extract numerical features. We'll extract:
1. **Pixel Statistics**: Mean, Standard Deviation, Min, Max
2. **Histogram Features**: Distribution of pixel intensities
3. **Flattened Pixel Values**: Direct pixel representation (for smaller images)

This converts images into feature vectors suitable for classical ML models.

In [ ]:
def extract_image_features(image_path, target_size=(64, 64)):
    """
    Extract statistical features from an image
    For real implementation, load actual image. 
    For demo, we generate synthetic features.
    """
    try:
        if isinstance(image_path, str) and not os.path.exists(image_path):
            # Generate synthetic features for demonstration
            np.random.seed(hash(image_path) % 10000)
            features = {
                'mean_intensity': np.random.uniform(50, 200),
                'std_intensity': np.random.uniform(20, 80),
                'min_intensity': np.random.uniform(0, 50),
                'max_intensity': np.random.uniform(200, 255),
                'median_intensity': np.random.uniform(80, 180),
            }
            # Add histogram features (10 bins)
            hist = np.random.dirichlet(np.ones(10)) * 100
            for i, val in enumerate(hist):
                features[f'hist_bin_{i}'] = val
            
            # Add some spatial features
            features['edge_density'] = np.random.uniform(0.1, 0.5)
            features['contrast'] = np.random.uniform(20, 100)
            
            return features
        else:
            # Real image loading code
            img = Image.open(image_path).convert('L')  # Convert to grayscale
            img = img.resize(target_size)
            img_array = np.array(img)
            
            features = {
                'mean_intensity': np.mean(img_array),
                'std_intensity': np.std(img_array),
                'min_intensity': np.min(img_array),
                'max_intensity': np.max(img_array),
                'median_intensity': np.median(img_array),
            }
            
            # Histogram features
            hist, _ = np.histogram(img_array, bins=10, range=(0, 256))
            hist = hist / hist.sum() * 100  # Normalize
            for i, val in enumerate(hist):
                features[f'hist_bin_{i}'] = val
            
            # Additional features
            features['edge_density'] = np.sum(np.abs(np.diff(img_array))) / img_array.size
            features['contrast'] = img_array.max() - img_array.min()
            
            return features
    except Exception as e:
        print(f"Error processing {image_path}: {e}")
        return None

print("Extracting features from images...")
print("This may take a few minutes...")

# Extract features for training set
train_features_list = []
for idx, path in enumerate(train_df['image_path']):
    if idx % 100 == 0:
        print(f"Processing training image {idx}/{len(train_df)}")
    features = extract_image_features(path)
    if features:
        train_features_list.append(features)

# Extract features for test set
test_features_list = []
for idx, path in enumerate(test_df['image_path']):
    if idx % 50 == 0:
        print(f"Processing test image {idx}/{len(test_df)}")
    features = extract_image_features(path)
    if features:
        test_features_list.append(features)

print(f"\n✓ Feature extraction completed")
print(f"Extracted {len(train_features_list)} training feature vectors")
print(f"Extracted {len(test_features_list)} test feature vectors")

## 6. Create Feature DataFrames

In [ ]:
# Convert features to DataFrame
X_train_features = pd.DataFrame(train_features_list)
X_test_features = pd.DataFrame(test_features_list)

# Add labels
y_train = train_df['label'].iloc[:len(X_train_features)].reset_index(drop=True)
y_test = test_df['label'].iloc[:len(X_test_features)].reset_index(drop=True)

print("Feature DataFrame Shape:")
print(f"Training features: {X_train_features.shape}")
print(f"Test features: {X_test_features.shape}")
print(f"\nFeature columns: {list(X_train_features.columns)}")
print(f"\nFirst few samples:")
print(X_train_features.head())

## 7. Encoding Categorical Data (Label Encoding)

In [ ]:
# Label Encoding: Convert NORMAL/PNEUMONIA to 0/1
label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

print("Label Encoding Mapping:")
print("=" * 60)
for idx, class_name in enumerate(label_encoder.classes_):
    print(f"{class_name} → {idx}")

print(f"\nEncoded training labels shape: {y_train_encoded.shape}")
print(f"Encoded test labels shape: {y_test_encoded.shape}")
print(f"Unique encoded values: {np.unique(y_train_encoded)}")

## 8. Feature Scaling

Standardize features to have mean=0 and std=1. This is crucial for algorithms like KNN, Logistic Regression, and others that are sensitive to feature scales.

In [ ]:
# Feature Scaling using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_features)
X_test_scaled = scaler.transform(X_test_features)

print("Feature Scaling Completed")
print("=" * 60)
print(f"Original feature mean: {X_train_features.mean().mean():.2f}")
print(f"Original feature std: {X_train_features.std().mean():.2f}")
print(f"\nScaled feature mean: {X_train_scaled.mean():.4f}")
print(f"Scaled feature std: {X_train_scaled.std():.4f}")

# Convert back to DataFrame for easier handling
X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=X_train_features.columns)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=X_test_features.columns)

print("\n✓ Feature scaling and preprocessing completed successfully")

## 9. Save Preprocessed Data

In [ ]:
# Save preprocessed data for use in the next notebook
import joblib

# Save scaler and label encoder
joblib.dump(scaler, '../models/feature_scaler.pkl')
joblib.dump(label_encoder, '../models/label_encoder.pkl')

# Save processed features
np.save('../data/X_train_scaled.npy', X_train_scaled)
np.save('../data/X_test_scaled.npy', X_test_scaled)
np.save('../data/y_train_encoded.npy', y_train_encoded)
np.save('../data/y_test_encoded.npy', y_test_encoded)

# Save feature names
pd.DataFrame({'feature_names': X_train_features.columns}).to_csv('../data/feature_names.csv', index=False)

print("✓ All preprocessed data saved successfully")
print("\nSaved files:")
print("  - models/feature_scaler.pkl")
print("  - models/label_encoder.pkl")
print("  - data/X_train_scaled.npy")
print("  - data/X_test_scaled.npy")
print("  - data/y_train_encoded.npy")
print("  - data/y_test_encoded.npy")
print("  - data/feature_names.csv")

## Summary

✅ **Completed Tasks:**
1. Data loading and exploration
2. Data cleaning with Pandas (handling missing values, duplicates)
3. Feature extraction from X-ray images
4. Label encoding for categorical target variable
5. Feature scaling using StandardScaler
6. Data saved for classical ML model training

**Next Steps:** Train classical ML models (Logistic Regression, Decision Tree, Random Forest, KNN, Naive Bayes) in the next notebook.